In [11]:
import requests
import pandas as pd
import time
from datetime import datetime
import os
from tqdm import tqdm

In [2]:
with open("../data/raw/app_ids_large_seed.txt", "r") as file:
    app_ids = [int(line.strip()) for line in file if line.strip()]

print(len(app_ids))
print(app_ids[:10])

483
[10, 20, 30, 40, 50, 60, 70, 80, 130, 220]


In [3]:
url = f"https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/?appid={730}"
response = requests.get(url)
data = response.json()

print(data.keys())

dict_keys(['appnews'])


In [4]:
data["appnews"]['newsitems']

[{'gid': '1831432155569878',
  'title': 'Counter-Strike 2 Update',
  'url': 'https://steamstore-a.akamaihd.net/news/externalpost/steam_community_announcements/1831432155569878',
  'is_external_url': True,
  'author': 'jo',
  'contents': "[p]\\[ MAPS ][/p][p]Cache[/p][list][*][p]Map-wide clipping fixes and geometry polish.[/p][/*][*][p]Fixed some spots where bomb would be unreachable when dropped.[/p][/*][*][p]Fixed dynamic shadows breaking in some spots.[/p][/*][*][p]Fixed some surface sound types.[/p][/*][/list][p][/p][p]\\[ ANIMGRAPH 2 ][/p][list][*][p]Fixed hand popping when counter-strafing with a grenade equipped.[/p][/*][/list][p][/p][p]\\[ MISC ][/p][list][*][p]Limit aim punch to 90 degrees.[/p][/*][*][p]Added secondary intersection trace for partially-occluded thirdperson weapons.[/p][/*][*][p]Adjusted ground smoothing at locations where sloped ground surfaces join with step-height transitions.[/p][/*][*][p]Fixed issue that caused defuse-cables from completely occluded players 

In [5]:
news_items = data["appnews"]["newsitems"]

unique_tags = set()

for post in news_items:
    tags = post.get("tags", [])
    for tag in tags:
        unique_tags.add(tag)

print(unique_tags)

{'mod_require_rereview', 'ModAct_1827889981_1775148648_4', 'ModAct_487176015_1777423238_4', 'ModAct_1803566692_1777416478_0', 'patchnotes', 'ModAct_1294976491_1775084755_0', 'ModAct_1870293249_1775093851_4', 'mod_reviewed', 'ModAct_1888979151_1773873692_0'}


In [6]:
data["appnews"]["newsitems"][0]

{'gid': '1831432155569878',
 'title': 'Counter-Strike 2 Update',
 'url': 'https://steamstore-a.akamaihd.net/news/externalpost/steam_community_announcements/1831432155569878',
 'is_external_url': True,
 'author': 'jo',
 'contents': "[p]\\[ MAPS ][/p][p]Cache[/p][list][*][p]Map-wide clipping fixes and geometry polish.[/p][/*][*][p]Fixed some spots where bomb would be unreachable when dropped.[/p][/*][*][p]Fixed dynamic shadows breaking in some spots.[/p][/*][*][p]Fixed some surface sound types.[/p][/*][/list][p][/p][p]\\[ ANIMGRAPH 2 ][/p][list][*][p]Fixed hand popping when counter-strafing with a grenade equipped.[/p][/*][/list][p][/p][p]\\[ MISC ][/p][list][*][p]Limit aim punch to 90 degrees.[/p][/*][*][p]Added secondary intersection trace for partially-occluded thirdperson weapons.[/p][/*][*][p]Adjusted ground smoothing at locations where sloped ground surfaces join with step-height transitions.[/p][/*][*][p]Fixed issue that caused defuse-cables from completely occluded players to als

In [7]:
readable_date = datetime.fromtimestamp(data["appnews"]["newsitems"][0]['date'])

print(readable_date)
print((data["appnews"]["newsitems"][0]['feed_type']))
print((data["appnews"]["newsitems"][0]['feedlabel']))


2026-04-30 18:46:49
1
Community Announcements


In [8]:

len(data["appnews"]["newsitems"])

20

In [9]:
def get_content_support(app_id):
    url = f"https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/?appid={app_id}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()

        news_items = data.get("appnews", {}).get("newsitems", [])
        if not news_items:
            return {
                "app_id": app_id,
                "latest_date": None,
                "feed_type": None,
                "feedlabel": None,
                "tags": None
            }

        latest_post = news_items[0]
        chosen_post = latest_post

        for post in news_items:
            if "patchnotes" in post.get("tags", []):
                chosen_post = post
                break

        return {
            "app_id": app_id,
            "latest_date": datetime.fromtimestamp(chosen_post["date"]),
            "feed_type": chosen_post.get("feed_type"),
            "feedlabel": chosen_post.get("feedlabel"),
            "tags": ", ".join(chosen_post.get("tags", []))
        }

    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return None

In [12]:
content_support = []

for app_id in tqdm(app_ids, desc="Collecting content support data"):
    row = get_content_support(app_id)
    content_support.append(row)
    time.sleep(1)

df_content = pd.DataFrame(content_support)
df_content.head()

,app_id,latest_date,feed_type,feedlabel,tags
0,10,2023-12-12 11:37:21,1.0,Community Announcements,patchnotes
1,20,2024-10-02 11:56:29,1.0,Community Announcements,patchnotes
2,30,2019-10-08 14:08:41,1.0,Community Announcements,patchnotes
3,40,2019-10-08 14:07:09,1.0,Community Announcements,patchnotes
4,50,2019-10-08 13:31:33,1.0,Community Announcements,patchnotes


In [13]:
print(df_content.shape)
print(df_content.info())

print(df_content.isnull().sum())


(483, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483 entries, 0 to 482
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   app_id       483 non-null    int64         
 1   latest_date  471 non-null    datetime64[ns]
 2   feed_type    471 non-null    float64       
 3   feedlabel    471 non-null    object        
 4   tags         471 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 19.0+ KB
None
app_id          0
latest_date    12
feed_type      12
feedlabel      12
tags           12
dtype: int64


In [14]:
print(df_content[df_content['tags'] == 'patchnotes'])

      app_id         latest_date  feed_type                feedlabel  \
0         10 2023-12-12 11:37:21        1.0  Community Announcements   
1         20 2024-10-02 11:56:29        1.0  Community Announcements   
2         30 2019-10-08 14:08:41        1.0  Community Announcements   
3         40 2019-10-08 14:07:09        1.0  Community Announcements   
4         50 2019-10-08 13:31:33        1.0  Community Announcements   
..       ...                 ...        ...                      ...   
474  2591280 2024-12-04 10:06:22        1.0  Community Announcements   
476  2646460 2026-04-29 03:02:31        1.0  Community Announcements   
477  2670630 2026-04-01 06:31:22        1.0  Community Announcements   
480  2767030 2026-04-15 06:52:48        1.0  Community Announcements   
482  2807960 2026-05-04 11:00:18        1.0  Community Announcements   

           tags  
0    patchnotes  
1    patchnotes  
2    patchnotes  
3    patchnotes  
4    patchnotes  
..          ...  
474  patc

In [15]:
df_content['tags'].unique()

array(['patchnotes', '', None,
       'patchnotes, auto_migrated, hide_library_overview, hide_library_detail',
       'patchnotes, hide_library_detail',
       'patchnotes, patchnotes, patchnotes, auto_migrated, hide_library_overview, hide_library_detail',
       'patchnotes, mod_reviewed, ModAct_871423085_1776787779_0, mod_require_rereview',
       'patchnotes, workshop',
       'patchnotes, mod_reviewed, ModAct_1691520782_1776960288_0, mod_require_rereview',
       'patchnotes, patchnotes, auto_migrated, hide_library_overview, hide_library_detail',
       'workshop',
       'mod_reviewed, ModAct_871423085_1774974206_0, mod_require_rereview',
       'patchnotes, mod_reviewed, ModAct_1751692459_1774586160_0, mod_require_rereview',
       'patchnotes, workshop, mod_reviewed, ModAct_1526778620_1777466710_0, ModAct_1894963720_1777477229_4, mod_require_rereview',
       'patchnotes, workshop, auto_migrated, hide_library_overview, hide_library_detail',
       'patchnotes, enable_steam_china

In [16]:
os.makedirs("../data/raw", exist_ok=True)
df_content.to_csv("../data/raw/steam_content_batch1.csv", index=False)

print("Saved raw content batch.")

Saved raw content batch.


In [17]:
df_content_clean = df_content.copy()

df_content_clean.columns = df_content_clean.columns.str.strip().str.lower()
df_content_clean = df_content_clean.drop_duplicates(subset="app_id")

os.makedirs("../data/cleaned", exist_ok=True)
df_content_clean.to_csv("../data/cleaned/steam_content_clean.csv", index=False)

print("Saved cleaned content data.")

Saved cleaned content data.
